# ComfyUI + Wan 2.1 1.3B on a free Kaggle / Colab GPU

This notebook gives you a **real, free** ComfyUI video backend for Movie Creator.
It installs ComfyUI, the VideoHelperSuite nodes, and the Wan 2.1 1.3B weights,
starts ComfyUI on port 8188, and prints a Cloudflare quick-tunnel URL you paste
into `COMFYUI_URL`.

**Honesty note:** nothing here fakes a render. If ComfyUI fails to start or the
weights are missing, the cell stops with the real error. A tunnel URL is only
printed once `/system_stats` actually answers 200.

Model files used (from `Comfy-Org/Wan-2.1`, the ComfyUI-repackaged weights):
`wan2.1_t2v_1.3B_fp16.safetensors`, `umt5_xxl_fp8_e4m3fn_scaled.safetensors`,
`wan_2.1_vae.safetensors`.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Walang GPU. Palitan ang runtime: GPU T4 (Colab) o Accelerator GPU (Kaggle)."
print("GPU:", torch.cuda.get_device_name(0))


## 2. Clone ComfyUI and the video helper nodes

In [ ]:
import os
%cd /content

if not os.path.isdir("ComfyUI"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
else:
    print("ComfyUI already cloned")

%cd /content/ComfyUI
!pip install -q -r requirements.txt

# VHS_VideoCombine (mp4 output) comes from this custom node pack.
if not os.path.isdir("custom_nodes/ComfyUI-VideoHelperSuite"):
    !git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git custom_nodes/ComfyUI-VideoHelperSuite
    !pip install -q -r custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
else:
    print("VideoHelperSuite already present")


## 3. Download the Wan 2.1 1.3B weights into the folders the workflow expects

In [ ]:
import os
from huggingface_hub import hf_hub_download

REPO = "Comfy-Org/Wan-2.1"
FILES = {
    "split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors": "models/diffusion_models",
    "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors": "models/text_encoders",
    "split_files/vae/wan_2.1_vae.safetensors": "models/vae",
}

for remote, local_dir in FILES.items():
    os.makedirs(local_dir, exist_ok=True)
    path = hf_hub_download(repo_id=REPO, filename=remote, local_dir=local_dir)
    size = os.path.getsize(path)
    assert size > 1_000_000, f"{path} is only {size} bytes - download failed"
    print(f"{path}  {size/1e9:.2f} GB")


## 4. Start ComfyUI on port 8188

The cell waits for a real `200` from `/system_stats` before returning.


In [ ]:
import subprocess, time, urllib.request, os

log = open("/content/comfyui.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT,
)

def stats(timeout=3):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=timeout) as r:
            return r.status, r.read().decode()
    except Exception as e:
        return None, str(e)

deadline = time.time() + 300
while time.time() < deadline:
    status, body = stats()
    if status == 200:
        print("ComfyUI is live")
        print(body[:500])
        break
    time.sleep(5)
else:
    print("ComfyUI did not start. Last 40 log lines:")
    print(open("/content/comfyui.log").read()[-4000:])
    raise RuntimeError("ComfyUI failed to start - see the log above")


## 5. Confirm the workflow's nodes and models are really there

In [ ]:
import json, urllib.request

with urllib.request.urlopen("http://127.0.0.1:8188/object_info") as r:
    info = json.load(r)

for node in ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode",
             "EmptyHunyuanLatentVideo", "KSampler", "VAEDecode", "VHS_VideoCombine"]:
    assert node in info, f"missing node {node} (restart ComfyUI after installing custom nodes)"
print("all workflow nodes present")

unets = info["UNETLoader"]["input"]["required"]["unet_name"][0]
clips = info["CLIPLoader"]["input"]["required"]["clip_name"][0]
vaes = info["VAELoader"]["input"]["required"]["vae_name"][0]
print("unet :", unets)
print("clip :", clips)
print("vae  :", vaes)
assert "wan2.1_t2v_1.3B_fp16.safetensors" in unets, "the 1.3B UNET is not visible to ComfyUI"
assert "umt5_xxl_fp8_e4m3fn_scaled.safetensors" in clips, "the text encoder is not visible"
assert "wan_2.1_vae.safetensors" in vaes, "the VAE is not visible"


## 6. Open a public tunnel and copy the URL

A Cloudflare quick tunnel needs no account. The printed `https://...trycloudflare.com`
is your `COMFYUI_URL`. Keep this notebook running while you render.


In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

import subprocess, time, re, urllib.request

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url = None
deadline = time.time() + 120
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)

if not url:
    raise RuntimeError("Walang tunnel URL na lumabas sa loob ng 120s. Tingnan ang output ng cloudflared.")

print("COMFYUI_URL =", url)
with urllib.request.urlopen(url + "/system_stats", timeout=30) as r:
    print("public /system_stats ->", r.status)
    assert r.status == 200, "the tunnel is up but ComfyUI did not answer through it"
print()
print("I-set ito sa .env ng Movie Creator:")
print(f"COMFYUI_URL={url}")
print("COMFYUI_VIDEO_WORKFLOW=./workflows/comfyui_video_api.json")
print("COMFYUI_CHECKPOINT=wan2.1_t2v_1.3B_fp16.safetensors")
print("PREFERRED_RENDER_ENGINE=comfyui")
print("ALLOW_PAID_CLOUD=0")


## 7. Smoke test: queue the real workflow once

This sends the repo's workflow with the real prompt and waits for the mp4. If it
fails, the error body from ComfyUI is printed as-is.


In [ ]:
import json, time, urllib.request, uuid

# Paste or upload workflows/comfyui_video_api.json to /content, then:
WORKFLOW_PATH = "/content/comfyui_video_api.json"
raw = open(WORKFLOW_PATH).read()
raw = (raw.replace("__PROMPT__", "a slow cinematic shot of rain over a rice field at dusk")
          .replace("__NEGATIVE__", "blurry, low quality, watermark")
          .replace("__SEED__", "42")
          .replace("__WIDTH__", "480")
          .replace("__HEIGHT__", "832")
          .replace("__FRAMES__", "33")
          .replace("__CHECKPOINT__", "wan2.1_t2v_1.3B_fp16.safetensors"))
workflow = json.loads(raw)

req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=json.dumps({"prompt": workflow, "client_id": str(uuid.uuid4())}).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req) as r:
    prompt_id = json.load(r)["prompt_id"]
print("queued prompt_id =", prompt_id)

deadline = time.time() + 1800
while time.time() < deadline:
    with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as r:
        hist = json.load(r)
    entry = hist.get(prompt_id)
    if entry:
        outs = entry.get("outputs", {})
        for node_out in outs.values():
            for key in ("gifs", "videos", "images"):
                for item in node_out.get(key, []):
                    if item.get("filename", "").endswith(".mp4"):
                        print("mp4 produced:", item["filename"])
                        raise SystemExit(0)
        print("outputs:", json.dumps(outs)[:600])
        raise RuntimeError("tapos ang prompt pero walang mp4 sa outputs")
    time.sleep(10)
raise RuntimeError("timeout waiting for the render")
